# Job Search API Smoke Test\n\nRun this notebook to quickly validate core API and RSS endpoints.

In [ ]:
import requests\nfrom pprint import pprint

In [ ]:
BASE_URL = \"http://127.0.0.1:8000\"\nprint(f\"Using BASE_URL={BASE_URL}\")

In [ ]:
def call_json(path, params=None, timeout=30):\n    url = BASE_URL.rstrip(\"/\") + path\n    r = requests.get(url, params=params or {}, timeout=timeout)\n    print(path, r.status_code)\n    r.raise_for_status()\n    return r.json()

In [ ]:
# Health + jobs JSON\nhealth = call_json(\"/health\")\njobs = call_json(\"/jobs\", {\"q\": \"data analyst\", \"days\": 7, \"limit\": 20})\nprint(\"health.ok:\", health.get(\"ok\"))\nprint(\"jobs.count:\", jobs.get(\"count\"))\nprint(\"first titles:\")\nfor j in (jobs.get(\"jobs\") or [])[:5]:\n    print(\"-\", j.get(\"title\"), \"|\", j.get(\"source\"))

In [ ]:
# RSS endpoint check\nrss_resp = requests.get(\n    BASE_URL.rstrip(\"/\") + \"/jobs/rss\",\n    params={\"q\": \"data analyst\", \"days\": 7, \"limit\": 20},\n    timeout=30,\n)\nprint(\"/jobs/rss\", rss_resp.status_code, rss_resp.headers.get(\"content-type\"))\nrss_resp.raise_for_status()\nrss_text = rss_resp.text\nprint(rss_text[:500])\nprint(\"contains <rss>:\", \"<rss\" in rss_text.lower())

In [ ]:
# Optional: rssjobs endpoint check (requires a valid rssjobs.app feed URL)\nfeed_url = \"\"  # paste generated rssjobs.app feed URL here\nif feed_url:\n    rssjobs = call_json(\"/rssjobs\", {\"feed_url\": feed_url, \"limit\": 50})\n    print(\"rssjobs.count:\", rssjobs.get(\"count\"))\nelse:\n    print(\"Skipped /rssjobs test (feed_url not set).\")

## Local Audit Box

This section runs the deeper local audit script and loads the saved results from `notebooks/output/job_api_audit_results.json`.

Use this when you want a fuller picture than the quick endpoint smoke test above.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'job_api_audit.py'
RESULT_PATH = REPO_ROOT / 'notebooks' / 'output' / 'job_api_audit_results.json'

print(f'Repo root: {REPO_ROOT}')
print(f'Script: {SCRIPT_PATH}')
print(f'Results: {RESULT_PATH}')

completed = subprocess.run([sys.executable, str(SCRIPT_PATH)], cwd=str(REPO_ROOT), capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f'Audit script failed with exit code {completed.returncode}')

In [ ]:
results = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
print('Generated at:', results['generated_at'])
print('Stored jobs:', results['stored_jobs_count'])
print('\nSummary:')
for item in results.get('summary', []):
    print('-', item)

In [ ]:
endpoint_checks = results['endpoint_checks']
for name, payload in endpoint_checks.items():
    print(f'\n[{name}] status={payload.get("status_code")} ok={payload.get("ok")}')
    if payload.get('json_ok'):
        body = payload.get('body', {})
        if isinstance(body, dict):
            print('count =', body.get('count'))
            print('error =', body.get('error'))
    else:
        print(payload.get('text_preview', '')[:250])

In [ ]:
print('Scraper checks:')
for check in results.get('scraper_checks', []):
    print('\nSources:', check.get('label'))
    print('ok:', check.get('ok'))
    print('count:', check.get('count'))
    if check.get('sample_titles'):
        print('sample titles:')
        for title in check['sample_titles'][:3]:
            print('  -', title)
    if check.get('error'):
        print('error:', check['error'])